# Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.lm_interface import LMInterface
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import Table, TableContext

In [3]:
llm_path = "model/weight/qwen25-7b"
embed_model_path = "model/weight/bge-base"
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)

# IR Indexing

In [4]:
DO_INDEXING = True

In [5]:
ir_system = LMInterface({
    "llm": llm_conductor.llm,
    "embed_model": llm_conductor.embed_model
})

In [ ]:
# Pneuma indexing
from processor.core.ir_system.ir_data_model import AbstractDocument


if DO_INDEXING:
    TABLES_PATH = "../../data_src/buysite"
    DATASET_NAME = "buysite"
    documents: list[AbstractDocument] = []
    metadata = pd.read_csv(f"{TABLES_PATH}/metadata.csv")
    for table_fname in os.listdir(f"{TABLES_PATH}/dataset"):
        try:
            table_name = f"{TABLES_PATH}/dataset/{table_fname}"
            table = pd.read_csv(table_name, nrows=100)
            documents.append(Table(
                doc_id=table_name,
                retriever_type=RetrieverType.PNEUMA,
                content=table,
                metadata={
                    "table_name": table_name,
                    "dataset_name": DATASET_NAME,
                }
            ))
            table_context = metadata[metadata["table"].str.endswith(table_fname[:-4].upper())].reset_index(drop=True)
            if len(table_context) > 0:
                documents.append(
                    TableContext(
                        doc_id=f"table_context_{table_name}",
                        retriever_type=RetrieverType.PNEUMA,
                        content=table_context["value"][0],
                        metadata={
                            "table_name": table_name,
                            "dataset_name": DATASET_NAME,
                            "type": "description",
                        }
                    )
                )
        except pd.errors.EmptyDataError:
            continue
    ir_system.index_documents(
        RetrieverType.PNEUMA,
        documents,
    )

Indexing documents on the retriever RetrieverType.PNEUMA.


100%|██████████| 24/24 [00:00<00:00, 126.96it/s]

Looking for an optimal batch size
Current mid batch size: 25
batch_messages: [[{'role': <Role.SYSTEM: 'system'>, 'content': 'A table, which represents <processor.core.ir_system.ir_data_model.TableContext object at 0x7f99b5820410>, has the following columns:\n/*\nASN_ID | ORG_ID | CARRIER | DOMAIN | SHIPMENT_CONTROL_ID | ELT_TS\n*/\nDescribe very briefly what the DOMAIN column represents. If not possible, simply state "No description."'}], [{'role': <Role.SYSTEM: 'system'>, 'content': 'A table, which represents <processor.core.ir_system.ir_data_model.TableContext object at 0x7f99b5820410>, has the following columns:\n/*\nASN_ID | ORG_ID | CARRIER | DOMAIN | SHIPMENT_CONTROL_ID | ELT_TS\n*/\nDescribe very briefly what the CARRIER column represents. If not possible, simply state "No description."'}], [{'role': <Role.SYSTEM: 'system'>, 'content': 'A table, which represents <processor.core.ir_system.ir_data_model.TableContext object at 0x7f99b5820410>, has the following columns:\n/*\nASN_ID


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Prompts: ['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n

# Scenario 1: Procurement

In [ ]:
# TODO